# GAN Example (PyTorch) — Generating Handwritten Digits (MNIST)

The same minimal **Generative Adversarial Network (GAN)** as `gan_mnist.ipynb`, but
implemented in **PyTorch**.

- **Generator** — turns a random noise vector into a fake `28x28` image.
- **Discriminator** — tries to tell real MNIST images from the generator's fakes.

> Needs `torch` and `torchvision`. Install with `pip install torch torchvision` if missing.
> Runs on CPU; automatically uses CUDA if available.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Load and prepare the data

Pixels are normalized to `[-1, 1]` to match the generator's `tanh` output.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                 # -> [0, 1]
    transforms.Normalize((0.5,), (0.5,)),  # -> [-1, 1]
])

BATCH_SIZE = 256
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print('Training images:', len(train_dataset))

## 2. The Generator

Takes a 100-dim noise vector and upsamples it to a `1x28x28` image with transposed
convolutions.

In [ ]:
NOISE_DIM = 100

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(NOISE_DIM, 7 * 7 * 256, bias=False),
            nn.BatchNorm1d(7 * 7 * 256),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 5, stride=1, padding=2, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(128, 64, 5, stride=2, padding=2, output_padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(64, 1, 5, stride=2, padding=2, output_padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 7, 7)
        return self.deconv(x)

generator = Generator().to(device)
print(generator)

## 3. The Discriminator

A small CNN that outputs a single logit (> 0 means "real").

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, 5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
        )

    def forward(self, x):
        return self.net(x)

discriminator = Discriminator().to(device)
print(discriminator)

## 4. Losses and optimizers

`BCEWithLogitsLoss` combines a sigmoid with binary cross-entropy, so the networks can
output raw logits.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
g_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4)
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-4)

In [ ]:
# A fixed noise seed so we can watch the SAME samples improve over time
num_examples = 16
seed = torch.randn(num_examples, NOISE_DIM, device=device)

def plot_samples(noise, epoch=None):
    generator.eval()
    with torch.no_grad():
        imgs = generator(noise).cpu() * 0.5 + 0.5  # back to [0, 1]
    generator.train()
    grid = make_grid(imgs, nrow=4).permute(1, 2, 0).numpy()
    plt.figure(figsize=(4, 4))
    plt.imshow(grid, cmap='gray')
    plt.axis('off')
    if epoch is not None:
        plt.title(f'Epoch {epoch}')
    plt.show()

## 5. Train

Each step we update the discriminator (on real + fake) and then the generator (trying to
fool the discriminator). `EPOCHS = 30` gives recognisable digits.

In [ ]:
EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    for real, _ in train_loader:
        real = real.to(device)
        bsz = real.size(0)
        ones = torch.ones(bsz, 1, device=device)
        zeros = torch.zeros(bsz, 1, device=device)

        # --- Train discriminator ---
        noise = torch.randn(bsz, NOISE_DIM, device=device)
        fake = generator(noise)
        d_real = discriminator(real)
        d_fake = discriminator(fake.detach())
        d_loss = criterion(d_real, ones) + criterion(d_fake, zeros)

        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # --- Train generator ---
        d_fake_for_g = discriminator(fake)
        g_loss = criterion(d_fake_for_g, ones)  # wants discriminator to say 'real'

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={g_loss.item():.3f}  disc_loss={d_loss.item():.3f}')
    if epoch % 5 == 0 or epoch == 1:
        plot_samples(seed, epoch)

## 6. Final generated digits

In [ ]:
plot_samples(torch.randn(num_examples, NOISE_DIM, device=device))

### Things to try

- Increase `EPOCHS` and watch the digits sharpen.
- Move to GPU automatically benefits from the `device` setup above.
- Save the generator with `torch.save(generator.state_dict(), 'gan_generator.pt')`.